## Setup and Connection

In [18]:
import json
import warnings
import pandas as pd
from datetime import datetime
from pathlib import Path
from pprint import pprint
from pymongo import MongoClient

## Load the data

In [2]:
# Connect to local MongoDB
client = MongoClient('localhost', 27017)

# Create/access database and collection
db = client['project']
collection = db['credit_applications']
print("Connected to MongoDB successfully!")

Connected to MongoDB successfully!


In [3]:
# Load the json file
current_dir = Path.cwd()
repo_root = current_dir.parent
json_path = repo_root / "data" / "raw_credit_applications.json"

with open(json_path, 'r') as file:
    data = json.load(file)

# Prepare the data by renaming the original '_id' 
# This prevents collisions while preserving the reference
for record in data:
    if '_id' in record:
        record['app_id'] = record.pop('_id')

# Clear the Collection
collection.delete_many({})

# Insert the data into a Collection
try:
    collection.insert_many(data)
    print(f"Successfully inserted {len(data)} documents.")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully inserted 502 documents.


## Quick Data Overview

In [4]:
# View a sample document to understand the structure
sample = collection.find_one()
pprint(sample)

{'_id': ObjectId('69a210b647fe8254e270b2aa'),
 'app_id': 'app_200',
 'applicant_info': {'date_of_birth': '2001-03-09',
                    'email': 'jerry.smith17@hotmail.com',
                    'full_name': 'Jerry Smith',
                    'gender': 'Male',
                    'ip_address': '192.168.48.155',
                    'ssn': '596-64-4340',
                    'zip_code': '10036'},
 'decision': {'loan_approved': False,
              'rejection_reason': 'algorithm_risk_score'},
 'financials': {'annual_income': 73000,
                'credit_history_months': 23,
                'debt_to_income': 0.2,
                'savings_balance': 31212},
 'processing_timestamp': '2024-01-15T00:00:00Z',
 'spending_behavior': [{'amount': 480, 'category': 'Shopping'},
                       {'amount': 790, 'category': 'Rent'},
                       {'amount': 247, 'category': 'Alcohol'}]}


## Audit Query 1: Find Duplicates

In [5]:
pipeline = [
    {
        "$group": {
            "_id": "$app_id",      # Group by the original ID field
            "count": {"$sum": 1},      # Count how many documents have this ID
            "docs": {"$push": "$_id"}  # Store the new ObjectIds for reference
        }
    },
    {
        "$match": {
            "count": {"$gt": 1}        # Only return those that appear more than once
        }
    }
]

duplicates = list(db.credit_applications.aggregate(pipeline))

print(f"Found {len(duplicates)} duplicate IDs.")
for entry in duplicates:
    print(f"ID: {entry['_id']} | Occurrences: {entry['count']}")

Found 2 duplicate IDs.
ID: app_001 | Occurrences: 2
ID: app_042 | Occurrences: 2


In [6]:
for entry in duplicates:
    app_id = entry['_id']
    # Fetch all versions from the database
    versions = list(db.credit_applications.find({"app_id": app_id}))
    
    print(f"\n{'='*50}")
    print(f"ANALYZING DUPLICATES FOR ID: {app_id}")
    print(f"{'='*50}")
    
    for i, doc in enumerate(versions):
        print(f"\n--- VERSION {i+1} (Internal _id: {doc['_id']}) ---")
        # Calculate 'completeness' (number of fields)
        field_count = len(doc.keys())
        print(f"Field Count: {field_count}")
        pprint(doc)


ANALYZING DUPLICATES FOR ID: app_001

--- VERSION 1 (Internal _id: 69a210b647fe8254e270b429) ---
Field Count: 6
{'_id': ObjectId('69a210b647fe8254e270b429'),
 'app_id': 'app_001',
 'applicant_info': {'date_of_birth': '1986-05-27',
                    'email': 'stephanie.nguyen47@mail.com',
                    'full_name': 'Stephanie Nguyen',
                    'gender': 'Female',
                    'ip_address': '10.121.120.213',
                    'ssn': '427-90-1892',
                    'zip_code': '90230'},
 'decision': {'loan_approved': False, 'rejection_reason': 'high_dti_ratio'},
 'financials': {'annual_income': 102000,
                'credit_history_months': 37,
                'debt_to_income': 0.42,
                'savings_balance': 0},
 'spending_behavior': [{'amount': 576, 'category': 'Fitness'}]}

--- VERSION 2 (Internal _id: 69a210b647fe8254e270b471) ---
Field Count: 7
{'_id': ObjectId('69a210b647fe8254e270b471'),
 'app_id': 'app_001',
 'applicant_info': {'email': '

For Application app_042 Joseph Lopez we should keep Version 2 because it contains the exact same personal and financial data as the first version but adds a Resubmission note which provides better context for the record.

For Application app_001 Stephanie Nguyen we should keep Version 1 because it contains critical personal information like the SSN and date of birth which is missing from the second version despite the error note.

The general rule for this is to prioritize the most complete record to ensure that no vital applicant information is lost.

In [7]:
# Remove the version of app_001 that has the 'DUPLICATE_ENTRY_ERROR' note
db.credit_applications.delete_one({
    "app_id": "app_001", 
    "notes": "DUPLICATE_ENTRY_ERROR"
})

# Remove the version of app_042 that is missing the 'RESUBMISSION' note
db.credit_applications.delete_one({
    "app_id": "app_042", 
    "notes": {"$exists": False}
})

print(f"Updated document count: {db.credit_applications.count_documents({})}")

Updated document count: 500


In [8]:
# Find duplicate SSNs - each person should appear only once!
pipeline_duplicates = [
    {
        "$group": {
            "_id": "$applicant_info.ssn",
            "count": {"$sum": 1},
            "names": {"$push": "$applicant_info.full_name"}
        }
    },
    {
        "$match": {
            "count": {"$gt": 1}
        }
    },
    {
        "$sort": {"count": -1}
    }
]

duplicates = list(collection.aggregate(pipeline_duplicates))

print(f"Found {len(duplicates)} duplicate SSNs:")
for dup in duplicates[:5]:  # Show first 5
    print(f"  SSN: {dup['_id']} - Count: {dup['count']} - Names: {dup['names']}")

Found 3 duplicate SSNs:
  SSN: None - Count: 4 - Names: ['Margaret Williams', 'Carolyn Martin', 'Larry Williams', 'Brandon Moore']
  SSN: 780-24-9300 - Count: 2 - Names: ['Susan Martinez', 'Gary Wilson']
  SSN: 937-72-8731 - Count: 2 - Names: ['Sandra Smith', 'Samuel Hill']


In [9]:
# Identify the application IDs for the records that are incomplete or conflicting

'''
invalid_app_ids = [
    "app_075", "app_120", "app_268", "app_165", # Missing SSNs
    "app_101", "app_234",                       # SSN Conflict for 937-72-8731
    "app_088", "app_016"                        # SSN Conflict for 780-24-9300
]

# Delete these records from the collection
result = db.credit_applications.delete_many({"app_id": {"$in": invalid_app_ids}})

print(f"Cleanup finished. Removed {result.deleted_count} invalid records.")
print(f"Final document count: {db.credit_applications.count_documents({})}")

'''

'\ninvalid_app_ids = [\n    "app_075", "app_120", "app_268", "app_165", # Missing SSNs\n    "app_101", "app_234",                       # SSN Conflict for 937-72-8731\n    "app_088", "app_016"                        # SSN Conflict for 780-24-9300\n]\n\n# Delete these records from the collection\nresult = db.credit_applications.delete_many({"app_id": {"$in": invalid_app_ids}})\n\nprint(f"Cleanup finished. Removed {result.deleted_count} invalid records.")\nprint(f"Final document count: {db.credit_applications.count_documents({})}")\n\n'

## Audit Query 2: Check Consistency

**Data Quality Dimension:** Consistency  
**Issue:** Same field having different encodings (e.g., "Male" vs "M")

In [10]:
# How many different gender values exist?
pipeline_gender_consistency = [
    {
        "$group": {
            "_id": "$applicant_info.gender",
            "count": {"$sum": 1}
        }
    },
    {
        "$sort": {"count": -1}
    }
]

gender_values = list(collection.aggregate(pipeline_gender_consistency))

print("Gender value distribution:")
print("Expected: 2 distinct values (Male, Female)")
print(f"Actual: {len(gender_values)} distinct values")
print()
for gv in gender_values:
    print(f"  '{gv['_id']}': {gv['count']} records")

Gender value distribution:
Expected: 2 distinct values (Male, Female)
Actual: 5 distinct values

  'Male': 194 records
  'Female': 193 records
  'F': 58 records
  'M': 53 records
  '': 2 records


In [11]:
# Standardize 'F' to 'Female'
collection.update_many(
    {"applicant_info.gender": "F"},
    {"$set": {"applicant_info.gender": "Female"}}
)

# Standardize 'M' to 'Male'
collection.update_many(
    {"applicant_info.gender": "M"},
    {"$set": {"applicant_info.gender": "Male"}}
)

# Handle missing values (e.g., set to 'Unknown' or drop)
collection.update_many(
    {"applicant_info.gender": ""},
    {"$set": {"applicant_info.gender": "Unknown"}}
)

UpdateResult({'n': 2, 'nModified': 2, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

In [12]:
# How many different gender values exist?
pipeline_gender_consistency = [
    {
        "$group": {
            "_id": "$applicant_info.gender",
            "count": {"$sum": 1}
        }
    },
    {
        "$sort": {"count": -1}
    }
]

gender_values = list(collection.aggregate(pipeline_gender_consistency))

print("Gender value distribution:")
print("Expected: 2 distinct values (Male, Female)")
print(f"Actual: {len(gender_values)} distinct values")
print()
for gv in gender_values:
    print(f"  '{gv['_id']}': {gv['count']} records")

Gender value distribution:
Expected: 2 distinct values (Male, Female)
Actual: 3 distinct values

  'Female': 251 records
  'Male': 247 records
  'Unknown': 2 records


In [13]:
# Find any DOB that does NOT follow the YYYY-MM-DD format
regex_pattern = "^\\d{4}-\\d{2}-\\d{2}$"

inconsistent_formats = list(collection.find({
    "applicant_info.date_of_birth": {"$not": {"$regex": regex_pattern}}
}))

print(f"Found {len(inconsistent_formats)} records with non-standard formats.")
for doc in inconsistent_formats:
    print(f"ID: {doc['app_id']} | DOB: {doc['applicant_info']['date_of_birth']}")

Found 161 records with non-standard formats.
ID: app_275 | DOB: 14/02/1982
ID: app_099 | DOB: 28/01/1990
ID: app_320 | DOB: 01/12/1978
ID: app_307 | DOB: 1990/07/26
ID: app_173 | DOB: 18/07/1979
ID: app_289 | DOB: 20/04/1979
ID: app_075 | DOB: 
ID: app_274 | DOB: 1986/11/20
ID: app_276 | DOB: 1995/05/07
ID: app_386 | DOB: 03/20/1968
ID: app_178 | DOB: 20/07/1997
ID: app_285 | DOB: 1987/06/28
ID: app_420 | DOB: 1988/04/06
ID: app_130 | DOB: 03/10/1981
ID: app_108 | DOB: 14/06/1975
ID: app_497 | DOB: 04/20/1994
ID: app_367 | DOB: 04/08/1979
ID: app_160 | DOB: 29/12/1982
ID: app_228 | DOB: 14/12/1987
ID: app_039 | DOB: 30/09/1978
ID: app_492 | DOB: 1994/03/03
ID: app_372 | DOB: 11/03/1967
ID: app_392 | DOB: 28/11/1998
ID: app_264 | DOB: 1996/04/07
ID: app_154 | DOB: 12/16/1985
ID: app_114 | DOB: 1991/03/01
ID: app_323 | DOB: 08/11/1981
ID: app_479 | DOB: 1983/11/08
ID: app_260 | DOB: 09/10/1967
ID: app_247 | DOB: 1999/06/16
ID: app_059 | DOB: 1992/11/21
ID: app_297 | DOB: 02/18/1983
ID: a

In [ ]:
#Standardize the date format
def standardize_dob_silent():
    updated_count = 0
    skipped_count = 0
    
    records = list(collection.find())
    
    # Temporarily silence specific parsing warnings for a cleaner output
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning)
        
        for doc in records:
            raw_dob = doc.get('applicant_info', {}).get('date_of_birth')
            if not raw_dob:
                continue
                
            try:
                # Use format='mixed' if using pandas 2.0+, otherwise flexible parsing
                # This handles '2001-03-09' and '09/03/2001' equally well
                clean_date = pd.to_datetime(raw_dob, dayfirst=True, errors='raise')
                standardized_dob = clean_date.strftime('%Y-%m-%d')
                
                collection.update_one(
                    {"_id": doc['_id']},
                    {"$set": {"applicant_info.date_of_birth": standardized_dob}}
                )
                updated_count += 1
                    
            except Exception as e:
                print(f"Could not parse DOB for {doc.get('app_id')}: {raw_dob}")
                skipped_count += 1

    print(f"Standardization Complete (Warnings Silenced):")
    print(f"- Successfully formatted: {updated_count} records")
    print(f"- Skipped (Unparseable): {skipped_count} records")

standardize_dob_silent()

Standardization Complete (Warnings Silenced):
- Successfully formatted: 496 records
- Skipped (Unparseable): 0 records


In [20]:
# Find any DOB that does NOT follow the YYYY-MM-DD format
regex_pattern = "^\\d{4}-\\d{2}-\\d{2}$"

inconsistent_formats = list(collection.find({
    "applicant_info.date_of_birth": {"$not": {"$regex": regex_pattern}}
}))

print(f"Found {len(inconsistent_formats)} records with non-standard formats.")
for doc in inconsistent_formats:
    print(f"ID: {doc['app_id']} | DOB: {doc['applicant_info']['date_of_birth']}")

Found 4 records with non-standard formats.
ID: app_075 | DOB: 
ID: app_120 | DOB: 
ID: app_350 | DOB: 
ID: app_165 | DOB: 


In [25]:
# Export the clean dataset for analysis
from bson import json_util

clean_data = list(collection.find({}, {'_id': 0}))

output_path = repo_root / "data" / "clean_credit_applications.json"

with open(output_path, 'w') as f:
    # Use json.dump with indent for readability (important for GitHub inspection)
    json.dump(clean_data, f, indent=2)

print(f"Successfully saved {len(clean_data)} records to {output_path}")

Successfully saved 500 records to c:\Users\artur\dego-project-team13\data\clean_credit_applications.json
